In [ ]:
!unzip "/content/archive (5).zip" -d /content/brain_dataset

Archive:  /content/archive (5).zip
  inflating: /content/brain_dataset/brain_tumor_dataset/no/1 no.jpeg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/10 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/11 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/12 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/13 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/14 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/15 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/17 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/18 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/19 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/2 no.jpeg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/20 no.jpg  
  inflating: /content/brain_dataset/brain_tumor_dataset/no/21 no.jpg  
  inflating: /content/brain_dataset/brain_

In [ ]:
!ls /content/brain_dataset

brain_tumor_dataset  no  yes


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
dataset = datasets.ImageFolder(
    root="/content/brain_dataset/brain_tumor_dataset",
    transform=transform
)

In [ ]:
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
print(dataset.classes)
print(dataset.class_to_idx)

['no', 'yes']
{'no': 0, 'yes': 1}


In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*56*56,128),
            nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 5

for epoch in range(epochs):

    correct = 0
    total = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # Accuracy calculation
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print("Epoch:", epoch+1,
          "Loss:", loss.item(),
          "Accuracy:", accuracy)

Epoch: 1 Loss: 0.24738547205924988 Accuracy: 87.35177865612648
Epoch: 2 Loss: 0.10073861479759216 Accuracy: 97.23320158102767
Epoch: 3 Loss: 0.03462624549865723 Accuracy: 98.81422924901186
Epoch: 4 Loss: 0.010033130645751953 Accuracy: 99.60474308300395
Epoch: 5 Loss: 0.015604582615196705 Accuracy: 100.0


In [ ]:
model.eval()

image, label = dataset[0]

with torch.no_grad():

    output = model(image.unsqueeze(0))

    pred = torch.argmax(output)

print("Prediction:", dataset.classes[pred])

Prediction: no
